<a href="https://colab.research.google.com/github/VivianCloud/BWSI-GirlsWhoCanLearn-Projects/blob/main/DocumentSkimmer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install pymupdf # used so python can read pdf files

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 MB 46.2 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 73.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pymupdf]


In [25]:
from google.colab import files # so you can upload files
import fitz

from sklearn.feature_extraction.text import TfidfVectorizer # this module lets you convert text into vectors
from sklearn.metrics.pairwise import cosine_similarity # this lets you compare how similar vectors are based on angle between them

In [8]:
uploaded = files.upload() # similar to input(), this asks user for a file

Saving stuyAcademicPolicies.pdf to stuyAcademicPolicies (2).pdf


In [21]:
# create the records list variable
records = [];

In [23]:
for filename in uploaded.keys():
  document = fitz.open(filename)

  for page_number, page in enumerate(document):
    text = page.get_text()
    #print(text)
    thisPage = (f"Page {page_number}", text)
    records.append(thisPage)

print(records)

[('Page 0', ' \n \n NEW YORK CITY PUBLIC SCHOOLS (NYCPS) \nStuyvesant High School \n 345 Chambers Street \n                                        New York, NY 10282 \n \n\u200b\nAcademic Policies \nSchool Year 2026 - 2027 \n \n\u200b\nInstructional Expectations\xa0 \n \nGrading Framework \n \nExplanation of Grades \n\u200b\nSchool Programming - Scope & Sequence \n \nHomework Policy \n \nThe Spiral of Communication \n \nTesting Policy \n \nAcademic Integrity \n \nAccommodation Room(s) \n \n \nFinal Version - 9/4/26 \n \n'), ('Page 1', ' \n \n NEW YORK CITY PUBLIC SCHOOLS (NYCPS) \nStuyvesant High School \n 345 Chambers Street \n                                        New York, NY 10282 \n \n \nInstructional Expectations\xa0\u200b\nSchool Year 2026 – 2027 \n\xa0 \nStuyvesant High School maintains a set of instructional expectations to promote learning, \naccountability, and academic integrity; to minimize stress for students, staff, and families; to establish \nclear communication proto

In [27]:
search_text = [text + " " for pageRef, text in records]

In [30]:
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2)) # stop_words removes the common words such as "the" and "a"
document_vectors = vectorizer.fit_transform(search_text)

In [37]:
def research(question, top_k=3, min_similarity=0.9):
    query_vector = vectorizer.transform([question])
    scores = cosine_similarity(query_vector, document_vectors).ravel() # then first one is similarity with first doc, second with second doc, and so forth
    print("QUESTION:", question)
    if scores.max() < min_similarity:
        print("No strong match found. The source collection may not cover this issue.")
        return
    # this (below) creates a list of the indexes of the ranked similarity scores named ranked
    ranked = [i for i in scores.argsort()[::-1] # .argsort() returns the indexes of the items in the list if the itey were rearranged in ascending order based on item value
              if scores[i] >= max(min_similarity, 0.15 * scores.max())][:top_k] # essentially only allows indexes to be appended if they meat the max score of 0.15 or 15% of the max, then takes the top_k
    for rank, index in enumerate(ranked, start=1): # loops through the items in the list. enumerate gives an items position and its value - so rank is index in list ranked and index is the index of that value in the original document list
        reference, text, tags = records[index] # retrieves the tuple of information from the original records
        print(f"\n{rank}. {reference} | similarity: {scores[index]:.2f}") # prints the info
        print("SOURCE:", text) # prints actual source text
    print("\nSource: U.S. National Archives | educational research only; not legal advice.")

In [38]:
QUESTION = "What is the grading policy for summative assessments?"
research(QUESTION)
# first run: no strong match found

QUESTION: What is the grading policy for summative assessments?
No strong match found. The source collection may not cover this issue.
